# SynBioCrow 2.4.15 — Bounded Evidence Enrichment
Upload only the 2.4.15 SOURCE ZIP. Uses the latest complete development_routes_* artifact directory and a persistent Drive evidence cache. No route generation; validation/evaluation records remain sealed.


In [ ]:
from google.colab import drive,files
from pathlib import Path
import hashlib,json,os,shutil,subprocess,sys,time,zipfile
drive.mount("/content/drive",force_remount=False)
T0=time.time(); REPO=Path("/content/SynBioCrow_2_4_15"); SRC=Path("/content/src2415")
BASE=Path("/content/drive/MyDrive/SynBioCrow/2.4"); OUT=BASE/"evidence_enrichment_2_4_15"
CACHE=BASE/"state_2_4_15"/"rhea_uniprot_evidence_cache.json"; OUT.mkdir(parents=True,exist_ok=True)
def st(p,m): print(f"[2.4.15] {p:5.1f}% | elapsed {(time.time()-T0)/60:5.1f} min | {m}",flush=True)
def run(cmd,cwd=None,timeout=7200):
    print("$"," ".join(map(str,cmd)),flush=True)
    e=os.environ.copy()
    if cwd:e["PYTHONPATH"]=str(cwd)+os.pathsep+e.get("PYTHONPATH","")
    q=subprocess.run(list(map(str,cmd)),cwd=cwd,env=e,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,timeout=timeout)
    print(q.stdout,flush=True)
    if q.returncode:raise subprocess.CalledProcessError(q.returncode,cmd,output=q.stdout)
st(3,"upload SOURCE ZIP only"); u=files.upload()
if len(u)!=1:raise RuntimeError("upload exactly one SOURCE ZIP")
name=next(iter(u)); z=Path("/content")/name; z.write_bytes(u[name])
st(8,"clone develop/2.4"); shutil.rmtree(REPO,ignore_errors=True)
run(["git","clone","--depth","1","--branch","develop/2.4","https://github.com/theiman112860/SynBioCrow.git",str(REPO)],timeout=600)
st(14,"overlay source"); shutil.rmtree(SRC,ignore_errors=True);SRC.mkdir()
with zipfile.ZipFile(z) as q:q.extractall(SRC)
for p in SRC.rglob("*"):
    if p.is_file() and p.name!="README_SOURCE.txt":
        d=REPO/p.relative_to(SRC);d.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(p,d)
st(20,"install lightweight core");run([sys.executable,"-m","pip","install","-q","-e",str(REPO)],timeout=900)
man=json.loads((REPO/"benchmarks/2.4/split_manifest_2_4_11.json").read_text())
dev=[r["record_id"] for r in man["records"] if r["split"]=="development"]; sealed=[r["record_id"] for r in man["records"] if r["split"]!="development"]
st(25,"bounded discovery of development artifacts")
dirs=sorted([p for p in BASE.glob("development_routes_*") if p.is_dir()],key=lambda p:p.stat().st_mtime,reverse=True)
chosen=next((d for d in dirs[:20] if all((d/f"{x}.json").is_file() for x in dev)),None)
if chosen is None:raise FileNotFoundError("complete development_routes_* directory not found")
print("INPUT",chosen);print("SEALED / NOT OPENED",sealed);print("CACHE",CACHE)
st(30,"enrich unique reaction evidence; persistent cache enables resume")
run([sys.executable,str(REPO/"scripts/v24_enrich_development_evidence_2_4_15.py"),
 "--input-dir",str(chosen),"--split-manifest",str(REPO/"benchmarks/2.4/split_manifest_2_4_11.json"),
 "--cache",str(CACHE),"--out-dir",str(OUT),"--max-new-edges","750"],cwd=REPO,timeout=7200)
st(88,"verify firewall and package")
s=json.loads((OUT/"evidence_enrichment_summary.json").read_text())
assert s["generation_invoked"] is False and s["validation_truth_accessed"] is False and s["evaluation_truth_accessed"] is False
assert set(s["sealed_nondevelopment_record_ids"])==set(sealed)
bundle=OUT/"SynBioCrow_2_4_15_BOUNDED_EVIDENCE_ENRICHMENT_ALL_OUTPUTS.zip"
with zipfile.ZipFile(bundle,"w",zipfile.ZIP_DEFLATED) as q:
    for p in sorted(OUT.iterdir()):
        if p.is_file() and p!=bundle:q.write(p,p.name)
st(100,"PASS");print("BUNDLE",bundle);print("CACHE",CACHE);print("SHA256",hashlib.sha256(bundle.read_bytes()).hexdigest());files.download(str(bundle))


## Resume / rescue
Rerun the main cell with the same SOURCE ZIP to continue from the persistent evidence cache. Or download the latest completed output below.


In [ ]:
from google.colab import files
from pathlib import Path
p=Path("/content/drive/MyDrive/SynBioCrow/2.4/evidence_enrichment_2_4_15/SynBioCrow_2_4_15_BOUNDED_EVIDENCE_ENRICHMENT_ALL_OUTPUTS.zip")
assert p.is_file(),p
files.download(str(p))
